# EDA Barrios - Análisis de datos geográficos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

### Carga y exploración de estructura

In [2]:
df = pd.read_parquet("../data/processed/01_consolidated/dataset_barrios.parquet")

print(f"\n ESTRUCTURA BARRIOS")
print(f"  Filas: {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Tamaño memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.head()



 ESTRUCTURA BARRIOS
  Filas: 131
  Columnas: 10
  Tamaño memoria: 0.02 MB


,CODDIS,NOMDIS,COD_BAR,NOMBRE,COD_DIS_TX,BARRIO_MAY,COD_DISBAR,NUM_BAR,BARRIO_MT,Area
0,1,Centro,11,Palacio,1,PALACIO,11,1,PALACIO,1.469906e+06
1,1,Centro,12,Embajadores,1,EMBAJADORES,12,2,EMBAJADORES,1.033724e+06
2,1,Centro,13,Cortes,1,CORTES,13,3,CORTES,5.918742e+05
3,1,Centro,14,Justicia,1,JUSTICIA,14,4,JUSTICIA,7.394135e+05
4,1,Centro,15,Universidad,1,UNIVERSIDAD,15,5,UNIVERSIDAD,9.480264e+05


### Análisis de nulos

In [3]:

nulos = df.isnull().sum()   
if nulos.sum() == 0:
    print(f"\n Sin valores nulos")
    
print(f"\n  Valores nulos por columna:")
nulos_df = pd.DataFrame({
        'Columna': nulos.index,
        'Nulos': nulos.values,
        '%': (nulos.values / len(df) * 100).round(2)
    })
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)
    
for _, row in nulos_df.iterrows():
        print(f"  {row['Columna']:30s} {row['Nulos']:>8,d}  ({row['%']:>6.2f}%)")


 Sin valores nulos

  Valores nulos por columna:


### Análisis de duplicados

In [7]:
duplicados_totales = df.duplicated().sum()
print(f"  Total (todas las columnas): {duplicados_totales:,}")

  Total (todas las columnas): 0


### Análisis de tipos de datos

In [4]:
print(f"\n Tipos de datos:")
tipos = df.dtypes.value_counts()
for tipo, cantidad in tipos.items():
    print(f"  {str(tipo):25s} {cantidad:3d} columnas")

df.info()


 Tipos de datos:
  int64                       5 columnas
  str                         4 columnas
  float64                     1 columnas
<class 'pandas.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CODDIS      131 non-null    int64  
 1   NOMDIS      131 non-null    str    
 2   COD_BAR     131 non-null    int64  
 3   NOMBRE      131 non-null    str    
 4   COD_DIS_TX  131 non-null    int64  
 5   BARRIO_MAY  131 non-null    str    
 6   COD_DISBAR  131 non-null    int64  
 7   NUM_BAR     131 non-null    int64  
 8   BARRIO_MT   131 non-null    str    
 9   Area        131 non-null    float64
dtypes: float64(1), int64(5), str(4)
memory usage: 15.8 KB


### Análisis de cobertura geografica

In [5]:
print(f"\n  COBERTURA GEOGRÁFICA:")
    
if 'CODDIS' in df.columns:
    distritos = df['CODDIS'].nunique()
    print(f"  Distritos únicos: {distritos}")
    
if 'NOMDIS' in df.columns:
    print(f"\n  Distritos:")
    for dist in sorted(df['NOMDIS'].unique()):
        count = len(df[df['NOMDIS'] == dist])
        print(f"    {dist:30s} {count:>3} barrios")
    
if 'COD_BAR' in df.columns:
    barrios = df['COD_BAR'].nunique()
    print(f"\n  Barrios únicos: {barrios}")


  COBERTURA GEOGRÁFICA:
  Distritos únicos: 21

  Distritos:
    Arganzuela                       7 barrios
    Barajas                          5 barrios
    Carabanchel                      7 barrios
    Centro                           6 barrios
    Chamartín                        6 barrios
    Chamberí                         6 barrios
    Ciudad Lineal                    9 barrios
    Fuencarral - El Pardo            8 barrios
    Hortaleza                        6 barrios
    Latina                           7 barrios
    Moncloa - Aravaca                7 barrios
    Moratalaz                        6 barrios
    Puente de Vallecas               6 barrios
    Retiro                           6 barrios
    Salamanca                        6 barrios
    San Blas - Canillejas            8 barrios
    Tetuán                           6 barrios
    Usera                            7 barrios
    Vicálvaro                        4 barrios
    Villa de Vallecas                3 barrio

### Análisis de área geográfica

In [6]:
print(f"\n ANÁLISIS DE ÁREA (m²):")
    
if 'Area' in df.columns:
    area_total_m2 = df['Area'].sum()
    area_total_km2 = area_total_m2 / 1e6
        
    print(f"  Área total Madrid: {area_total_m2:>15,.0f} m²")
    print(f"  Área total Madrid: {area_total_km2:>15,.1f} km²")
        
    print(f"\n  Estadísticas de área por barrio:")
    print(f"    Media: {df['Area'].mean():>15,.0f} m²")
    print(f"    Mediana: {df['Area'].median():>13,.0f} m²")
    print(f"    Min: {df['Area'].min():>17,.0f} m² ({df[df['Area'] == df['Area'].min()]['NOMBRE'].values[0]})")
    print(f"    Max: {df['Area'].max():>17,.0f} m² ({df[df['Area'] == df['Area'].max()]['NOMBRE'].values[0]})")
    print(f"    Desvío: {df['Area'].std():>12,.0f} m²")
        
    # Top 10 barrios por área
    print(f"\n  Top 10 barrios por área:")
    top_barrios = df.nlargest(10, 'Area')[['NOMBRE', 'NOMDIS', 'Area']]
    for idx, (_, row) in enumerate(top_barrios.iterrows(), 1):
        print(f"    {idx:2d}. {row['NOMBRE']:30s} ({row['NOMDIS']:25s}) {row['Area']:>15,.0f} m²")
        
    # Bottom 5 barrios
    print(f"\n  Bottom 5 barrios por área:")
    bottom_barrios = df.nsmallest(5, 'Area')[['NOMBRE', 'NOMDIS', 'Area']]
    for idx, (_, row) in enumerate(bottom_barrios.iterrows(), 1):
            print(f"    {idx}. {row['NOMBRE']:30s} ({row['NOMDIS']:25s}) {row['Area']:>15,.0f} m²")


 ANÁLISIS DE ÁREA (m²):
  Área total Madrid:     604,014,551 m²
  Área total Madrid:           604.0 km²

  Estadísticas de área por barrio:
    Media:       4,610,798 m²
    Mediana:     1,363,212 m²
    Min:           248,883 m² (Atalaya)
    Max:       187,583,248 m² (El Pardo)
    Desvío:   17,120,417 m²

  Top 10 barrios por área:
     1. El Pardo                       (Fuencarral - El Pardo    )     187,583,248 m²
     2. Casco Histórico de Vallecas    (Villa de Vallecas        )      42,989,207 m²
     3. Aeropuerto                     (Barajas                  )      29,664,731 m²
     4. El Goloso                      (Fuencarral - El Pardo    )      26,493,113 m²
     5. Casco Histórico de Vicálvaro   (Vicálvaro                )      21,076,000 m²
     6. Casa de Campo                  (Moncloa - Aravaca        )      17,476,000 m²
     7. Valdefuentes                   (Hortaleza                )      17,023,150 m²
     8. Ciudad Universitaria           (Moncloa - Aravaca  

### Análisis de distrito

In [7]:
print(f"\n ÁREA POR DISTRITO:")
    
if 'NOMDIS' in df.columns and 'Area' in df.columns:
    area_dist = df.groupby('NOMDIS')['Area'].sum().sort_values(ascending=False)
        
    print(f"\n  Distritos por área total:")
    for dist, area in area_dist.items():
        area_km2 = area / 1e6
        print(f"    {dist:30s} {area:>15,.0f} m² ({area_km2:>7.1f} km²)")


 ÁREA POR DISTRITO:

  Distritos por área total:
    Fuencarral - El Pardo              237,838,267 m² (  237.8 km²)
    Villa de Vallecas                   51,410,558 m² (   51.4 km²)
    Moncloa - Aravaca                   46,586,950 m² (   46.6 km²)
    Barajas                             41,716,343 m² (   41.7 km²)
    Vicálvaro                           34,841,930 m² (   34.8 km²)
    Hortaleza                           27,623,984 m² (   27.6 km²)
    Latina                              25,410,388 m² (   25.4 km²)
    San Blas - Canillejas               22,289,895 m² (   22.3 km²)
    Villaverde                          20,194,323 m² (   20.2 km²)
    Puente de Vallecas                  14,968,606 m² (   15.0 km²)
    Carabanchel                         14,048,316 m² (   14.0 km²)
    Ciudad Lineal                       11,425,678 m² (   11.4 km²)
    Chamartín                            9,175,482 m² (    9.2 km²)
    Usera                                7,777,717 m² (    7.8 km²

### Análisis de nombres

In [8]:
print(f"\n  VALIDACIÓN DE NOMBRES:")
    
if 'NOMBRE' in df.columns:
    nombres_unicos = df['NOMBRE'].nunique()
    print(f"  NOMBRE (columna base): {nombres_unicos} únicos")
    
if 'BARRIO_MAY' in df.columns:
    barrios_may_unicos = df['BARRIO_MAY'].nunique()
    print(f"  BARRIO_MAY (mayúsculas): {barrios_may_unicos} únicos")
    
if 'BARRIO_MT' in df.columns:
    barrios_mt_unicos = df['BARRIO_MT'].nunique()
    print(f"  BARRIO_MT (título): {barrios_mt_unicos} únicos")
    
print(f"\n  Formato de nombres:")
if 'NOMBRE' in df.columns:
    print(f"    Ejemplos:")
    for nombre in df['NOMBRE'].head(5):
        print(f"      • {nombre}")



  VALIDACIÓN DE NOMBRES:
  NOMBRE (columna base): 131 únicos
  BARRIO_MAY (mayúsculas): 131 únicos
  BARRIO_MT (título): 131 únicos

  Formato de nombres:
    Ejemplos:
      • Palacio
      • Embajadores
      • Cortes
      • Justicia
      • Universidad
